# High-Level API
> This is the module for easy use of RevChem to conduct basic tasks

In [1]:
#| default_exp easy_api

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
# | export
import polars as pl

from pathlib import Path
from RevChem.data.export import (
    match_tobii_to_realeye_groups,
    pipeline_raw_realeye_to_timed_dataframe,
)
from RevChem.data.segmentation import (
    join_chunks_as_segments,
    render_point_stream_video_with_opencv,
    select_trial,
)
from RevChem.tobii import COLUMN_RENAMING_TOBII_TO_CSV


In [ ]:
#| export
def generate_participant_videos_from_files(
    realeye_csv_path: str,
    tobii_tsv_dir_path: str,
    video_output_dir: str,
    *,
    use_cache: bool = True
):
    """Uses RevChem low-level modules to produce an MP4 from the provided data

    Args:
        realeye_csv_path: absolute or relative path to the "raw gazes" CSV
        tobii_tsv_dir_path: absolute or relative path to the directory of all associated Tobii data
        video_output_path: 
        use_cache: _description_. Defaults to True.
    """
    # TODO

In [12]:
#| export
from RevChem.common import datetime_to_stamp
from RevChem.realeye import read_realeye_csv
from RevChem.data.export import (
    load_tobii_individual,
    rekeyed, # needed for the patch to GroupFrames, used on 
)
from RevChem.tobii import COLUMNS_TOBII


def generate_single_participant_video_from_files(
    realeye_csv_path: str,
    tobii_tsv_path: str,
    video_output_dir: str,
    *,
    realeye_stimulus_items_in_order: list[str],
    stimuli_image_paths: list[str | Path],
    use_cache: bool = True,
):
    """Uses RevChem low-level modules to produce an MP4 from the provided data

    Args:
        realeye_csv_path: Path to the raw-gazes.csv
        tobii_tsv_path: Path to a single participant's trial. Name will be used to name output video
        video_output_dir: Directory with write permissions, where the MP4 will be written
        use_cache: Whether to save the data to disk. Defaults to True.
        realeye_stimulus_items_in_order: Identifiers of the stimuli provided in/through RealEye, in the order
            the are presented.
            Originally, we intended to do this ordering programmatically, but were thwarted by the fact that
            RealEye's output rows are not ordered by "display order" but the order in which the stimuli were
            loaded into the system. For those of us that show the same stimuli multiple times, this makes
            downstream data near-useless, no matter how we true to infer the timing of RealEye data.
            (See those functions for more of that story.) The easiest way to comply with this is to go into the
            RealEye UI, inspect the order in which you are showing your images, and to copy-paste in-order the
            stimuli "Item ID"s as they appear. The order in the exported raw-gazes(-denoised).csv are not
            guaranteed to be accurate.
        stimuli_image_paths: Paths to the images to be shown as stimuli, in order.
            The easiest way to comply with this is to (very easily) download the stimuli to an isolated folder
            on your test machine, name then in lexicographical order (such that when you "Sort by Name"
            in File Explorer or Finder, they are in the order you would show a participant) and then load them
            with
            >>> from pathlib import Path
            >>> stimuli_image_paths = sorted(Path("my isolated stimuli directory").glob("*.jpg"))
    """
    # 1. Load and process RealEye data
    re_df = read_realeye_csv(realeye_csv_path)
    re_timed_dfs = pipeline_raw_realeye_to_timed_dataframe(
        re_df,
        item_ids_in_order=realeye_stimulus_items_in_order,
        dt_timestamp_col="timestamp",
    )

    # 2. Load and process Tobii data
    tobii_df = load_tobii_individual(
        tobii_tsv_path, columns=COLUMNS_TOBII, renaming=COLUMN_RENAMING_TOBII_TO_CSV,
        # clean_func=lambda fname: fname
    )

    # 3. Match Tobii and RealEye data
    matched_data = match_tobii_to_realeye_groups(
        [tobii_df],
        re_timed_dfs.rekeyed(
            lambda _, dfs: min(
                df["timestamp"].min() for df in dfs
            )  # smallest time is the RE start time
        ),
    )

    # 4. Join data into segments
    segmented_data = join_chunks_as_segments(matched_data, null_handling="complex")

    # 5. Render video
    trial_data = select_trial(Path(tobii_tsv_path).name, segmented_data)
    run_output_dir_name = f"RevChem-Output-{datetime_to_stamp()}"
    render_point_stream_video_with_opencv(
        stimuli_image_paths,
        trial_data,
        output_file_name=Path(video_output_dir, run_output_dir_name, trial_data.trial_name_or_id).expanduser().as_posix(),
        export_fps=60,
        n_images_to_show=4,
    )


## Simple workflow: Load your data and view it on your stimuli
Suppose you have just exported your participants' RealEye and Tobii data to your computer and you would like to visualize these two together.
RevChem can generate a video (`.mp4`) of both time series over your stimuli!

Requirements
- Realeye data must be...
    - be the "Raw Gazes" `.csv` file. Not the "Denoised" or "Smoothed" options.
    - support for the "Raw Gazes Denoised" and "Raw Gazes Denoised, Normalized and Split" is questionable
- Tobii data must...
    - include the Gazepoint Coordinates (X, Y) as a feature output.
    - include Recording Start Time and Recording Start Date, **both** localized and UTC.
        - We need this for temporally aligning the Realeye and Tobii recordings
    - be exported as a directory of `.tsv` files, **not `.xslx`**.
- Write permissions in the given output directory

In [13]:
from RevChem.data.export import _INFERRED_CU_CUA_CORRECT_ITEM_IDS

generate_single_participant_video_from_files(
    "~/dev/RevChemData/2025-05-14-Data_Export/RealEye/raw-gazes.csv",
    "~/dev/RevChemData/2025-05-14-Data_Export/Tobii-All-Snapshot/1.Realeye1,2,3 2025-03-05_Blastoise.tsv",
    "~/dev/RevChemData/2025-08-01-high-level-outputs/",
    realeye_stimulus_items_in_order=_INFERRED_CU_CUA_CORRECT_ITEM_IDS,
    stimuli_image_paths=sorted(Path("/Users/stephen/dev/RevChem-Stimuli/jpegs").glob("*.jpg")),
)

Got a 7-tuple: sextuple = [857, 857, 4249, 0, 944, 580, 1]
Got a 7-tuple: sextuple = [1416, 209, 27024, 0, 1919, 1037, 1]
Got a 7-tuple: sextuple = [545, 1043, 3167, 0, 665, 932, 1]
Got a 7-tuple: sextuple = [237, 267, 40087, 0, 1172, 701, 1]
Got a 7-tuple: sextuple = [804, 475, 40287, 0, 1172, 701, 1]
Pre-loading images...
Images loaded.
Processing trial 1/4...
Processing trial 2/4...
Processing trial 3/4...
Processing trial 4/4...
Finished writing video to /Users/stephen/dev/RevChemData/2025-08-01-high-level-outputs/RevChem-Output-202508011250/1.Realeye1,2,3 2025-03-05_Blastoise.tsv.mp4


In [ ]:
generate_participant_videos_from_files(
    "~/dev/RevChem_Data/{FOLDER_NAME}/raw-gazes.csv",
    "~/dev/RevChem_Data/{FOLDER_NAME}/Tobii-export/my_trial_01.tsv",
    "~/dev/RevChem_Data/2025-07-29-high-level-outputs/",
)